source of data: [Instituto Nacional de Estadística](https://www.ine.es/jaxiT3/Tabla.htm?t=1448&L=1), [CountryEconomy.com](https://countryeconomy.com/demography/life-expectancy/spain-autonomous-communities)<br>
wiki > [List of Spanish provinces by life expectancy](https://en.wikipedia.org/wiki/List_of_Spanish_provinces_by_life_expectancy) <i>([alternative](https://en.wikipedia.org/wiki/User:Lady3mlnm/List_of_Spanish_provinces_by_life_expectancy_(alternative)))</i>, [Political divisions of Spain](https://en.wikipedia.org/wiki/Political_divisions_of_Spain), [Provinces of Spain](https://en.wikipedia.org/wiki/Provinces_of_Spain)<br>
wiki > [Продолжительность жизни в провинциях Испании](https://ru.wikipedia.org/wiki/Продолжительность_жизни_в_провинциях_Испании), [Административное деление Испании](https://ru.wikipedia.org/wiki/Административное_деление_Испании), [Провинции Испании](https://ru.wikipedia.org/wiki/Провинции_Испании)

In [2]:
import pandas as pd
import math
import re

import sys
sys.path.append("..")
import mal_moduls_private.mal_total as mal

In [3]:
pd.options.display.max_columns = 30

docs: [standard-encodings](https://docs.python.org/3/library/codecs.html#standard-encodings)<br>
docs: [docs: error-handlers](https://docs.python.org/3/library/codecs.html#error-handlers)

In [5]:
# load stats about longevity per year
# df = pd.read_csv('data/Life expectancy in Spain -ac.csv', sep='\t', decimal=',')  # , encoding='mbcs', encoding_errors='replace' ascii
df = pd.read_csv('data/Life expectancy in Spain -ac.csv', sep='\t', encoding='mbcs')  # encoding_errors='replace' ascii

df.head(2)

,Autonomous Communities and Cities,Sex,Period,Total
0,01 Andalucía,Both genders,2024,82.82
1,01 Andalucía,Both genders,2023,82.49


In [6]:
# remove numbers before provinces names
df['Autonomous Communities and Cities'] = df['Autonomous Communities and Cities'].map(lambda st:st[3:])

# set provinces names as index
df.set_index('Autonomous Communities and Cities', drop=True, inplace=True)
df.index.name = ''

df.head(10)

,Sex,Period,Total
,,,
Andalucía,Both genders,2024,82.82
Andalucía,Both genders,2023,82.49
Andalucía,Both genders,2022,81.86
Andalucía,Both genders,2021,81.46
Andalucía,Both genders,2020,81.51
Andalucía,Both genders,2019,82.20
Andalucía,Both genders,2018,81.73
Andalucía,Both genders,2017,81.81
Andalucía,Both genders,2016,81.83


In [7]:
df = pd.concat([df[(df.Sex == 'Both genders') & (df.Period == 2014)].Total,
                df[(df.Sex == 'Both genders') & (df.Period == 2019)].Total,
                df[(df.Sex == 'Both genders') & (df.Period == 2020)].Total,
                df[(df.Sex == 'Both genders') & (df.Period == 2021)].Total,
                df[(df.Sex == 'Both genders') & (df.Period == 2022)].Total,
                df[(df.Sex == 'Both genders') & (df.Period == 2023)].Total,
                df[(df.Sex == 'Both genders') & (df.Period == 2024)].Total,
                df[(df.Sex == 'Males') & (df.Period == 2014)].Total,
                df[(df.Sex == 'Males') & (df.Period == 2019)].Total,
                df[(df.Sex == 'Males') & (df.Period == 2024)].Total,
                df[(df.Sex == 'Females') & (df.Period == 2014)].Total,
                df[(df.Sex == 'Females') & (df.Period == 2019)].Total,
                df[(df.Sex == 'Females') & (df.Period == 2024)].Total
               ],
                  axis='columns',
                  keys=['2014', '2019', '2020', '2021', '2022', '2023', '2024', 'm_2014', 'm_2019', 'm_2024', 'f_2014', 'f_2019', 'f_2024'])

df.sort_values(by=['2024', 'm_2024'], ascending=False, inplace=True)

df.rename(index = {
    'Andalucía' : 'Andalusia',
    'Aragón' : 'Aragon',
    'Asturias, Principado de' : 'Asturias',
    'Balears, Illes' : 'Balearic Islands',
    'Canarias' : 'Canary Islands',
    'Castilla - La Mancha' : 'Castilla La Mancha',
    'Castilla y León' : 'Castile and León',
    'Cataluña' : 'Catalonia',
    'Comunitat Valenciana' : 'Comunidad Valenciana',
    'Extremadura' : 'Estremadura',
    'Madrid, Comunidad de' : 'Madrid',
    'Murcia, Región de' : 'Murcia',
    'Navarra, Comunidad Foral de' : 'Navarre',
    'País Vasco' : 'Basque Country',
    'Rioja, La' : 'La Rioja'
}, inplace=True)

print(len(df))
df.head()

19


,2014,2019,2020,2021,2022,2023,2024,m_2014,m_2019,m_2024,f_2014,f_2019,f_2024
,,,,,,,,,,,,,
Madrid,84.20,84.93,82.27,84.56,84.76,85.39,85.58,81.44,82.33,83.07,86.59,87.19,87.74
Castile and León,83.60,84.20,82.53,83.94,83.68,84.54,84.67,80.82,81.52,82.08,86.44,86.94,87.28
Navarre,83.45,84.55,83.33,84.26,83.85,84.79,84.66,80.63,82.10,81.96,86.26,86.92,87.33
Basque Country,83.35,83.98,83.18,83.70,83.49,84.42,84.64,80.38,81.05,81.91,86.12,86.76,87.20
Aragon,82.85,83.88,82.38,83.25,83.06,84.07,84.24,80.03,81.26,81.71,85.72,86.48,86.72


<br />
<br />

Add data about Spain as a whole ([all](https://www.ine.es/consul/serie.do?d=true&s=TM2177), [male](https://www.ine.es/consul/serie.do?d=true&s=TM1449), [female](https://www.ine.es/consul/serie.do?d=true&s=TM721))

In [9]:
df_country_all = pd.read_csv('data/Life expectancy in Spain as a whole -all.csv', sep='\t', usecols=['PERIOD', 'VALUE'], index_col='PERIOD')
df_country_male = pd.read_csv('data/Life expectancy in Spain as a whole -male.csv', sep='\t', usecols=['PERIOD', 'VALUE'], index_col='PERIOD')
df_country_female = pd.read_csv('data/Life expectancy in Spain as a whole -female.csv', sep='\t', usecols=['PERIOD', 'VALUE'], index_col='PERIOD')

In [10]:
df_country = pd.DataFrame({'2014'  : [df_country_all.loc[2014, 'VALUE']],
                           '2019'  : [df_country_all.loc[2019, 'VALUE']],
                           '2020'  : [df_country_all.loc[2020, 'VALUE']],
                           '2021'  : [df_country_all.loc[2021, 'VALUE']],
                           '2022'  : [df_country_all.loc[2022, 'VALUE']],
                           '2023'  : [df_country_all.loc[2023, 'VALUE']],
                           '2024'  : [df_country_all.loc[2024, 'VALUE']],
                           'm_2014'  : [df_country_male.loc[2014, 'VALUE']],
                           'm_2019'  : [df_country_male.loc[2019, 'VALUE']],
                           'm_2024'  : [df_country_male.loc[2024, 'VALUE']],
                           'f_2014': [df_country_female.loc[2014, 'VALUE']],
                           'f_2019': [df_country_female.loc[2019, 'VALUE']],
                           'f_2024': [df_country_female.loc[2024, 'VALUE']],  
                          }, index=['Spain']).round(2)

df_country

,2014,2019,2020,2021,2022,2023,2024,m_2014,m_2019,m_2024,f_2014,f_2019,f_2024
Spain,82.91,83.53,82.28,83.03,83.08,83.77,84.01,80.1,80.78,81.38,85.63,86.19,86.53


In [11]:
df = pd.concat([df_country, df])

print(len(df))
df.head()

20


,2014,2019,2020,2021,2022,2023,2024,m_2014,m_2019,m_2024,f_2014,f_2019,f_2024
Spain,82.91,83.53,82.28,83.03,83.08,83.77,84.01,80.10,80.78,81.38,85.63,86.19,86.53
Madrid,84.20,84.93,82.27,84.56,84.76,85.39,85.58,81.44,82.33,83.07,86.59,87.19,87.74
Castile and León,83.60,84.20,82.53,83.94,83.68,84.54,84.67,80.82,81.52,82.08,86.44,86.94,87.28
Navarre,83.45,84.55,83.33,84.26,83.85,84.79,84.66,80.63,82.10,81.96,86.26,86.92,87.33
Basque Country,83.35,83.98,83.18,83.70,83.49,84.42,84.64,80.38,81.05,81.91,86.12,86.76,87.20


In [12]:
df.insert(loc=1 , column='2014→2019', value=df['2019']-df['2014'])
df.insert(loc=3 , column='2019→2020', value=df['2020']-df['2019'])
df.insert(loc=5, column='2020→2021', value=df['2021']-df['2020'])
df.insert(loc=7, column='2021→2022', value=df['2022']-df['2021'])
df.insert(loc=9, column='2022→2023', value=df['2023']-df['2022'])
df.insert(loc=11, column='2023→2024', value=df['2024']-df['2023'])
df.insert(loc=13, column='2019→2024', value=df['2024']-df['2019'])
df.insert(loc=14, column='2014→2024', value=df['2024']-df['2014'])
df['f-m_2014'] = df['f_2014']-df['m_2014']
df['f-m_2019'] = df['f_2019']-df['m_2019']
df['f-m_2024'] = df['f_2024']-df['m_2024']

df.head()

,2014,2014→2019,2019,2019→2020,2020,2020→2021,2021,2021→2022,2022,2022→2023,2023,2023→2024,2024,2019→2024,2014→2024,m_2014,m_2019,m_2024,f_2014,f_2019,f_2024,f-m_2014,f-m_2019,f-m_2024
Spain,82.91,0.62,83.53,-1.25,82.28,0.75,83.03,0.05,83.08,0.69,83.77,0.24,84.01,0.48,1.10,80.10,80.78,81.38,85.63,86.19,86.53,5.53,5.41,5.15
Madrid,84.20,0.73,84.93,-2.66,82.27,2.29,84.56,0.20,84.76,0.63,85.39,0.19,85.58,0.65,1.38,81.44,82.33,83.07,86.59,87.19,87.74,5.15,4.86,4.67
Castile and León,83.60,0.60,84.20,-1.67,82.53,1.41,83.94,-0.26,83.68,0.86,84.54,0.13,84.67,0.47,1.07,80.82,81.52,82.08,86.44,86.94,87.28,5.62,5.42,5.20
Navarre,83.45,1.10,84.55,-1.22,83.33,0.93,84.26,-0.41,83.85,0.94,84.79,-0.13,84.66,0.11,1.21,80.63,82.10,81.96,86.26,86.92,87.33,5.63,4.82,5.37
Basque Country,83.35,0.63,83.98,-0.80,83.18,0.52,83.70,-0.21,83.49,0.93,84.42,0.22,84.64,0.66,1.29,80.38,81.05,81.91,86.12,86.76,87.20,5.74,5.71,5.29


<br />
<br />

In [14]:
# just for interest, explore results: determine regions with max and min values, and also look at specific regions
mal.min_and_max_values(df[['2014', '2014→2019', '2019', '2019→2024', '2024', '2014→2024']],
                       row_center=['Spain'], nmb=5, max_lng=11)

Number of records: 20


,2014,2014→2019,2019,2019→2024,2024,2014→2024
max,84.2 -Madrid,1.1 -Navarre,84.93 -Madrid,1.99 -Melilla,85.58 -Madrid,2.63 -Melilla
max_2,83.78 -La Rioja,1.03 -Aragon,84.55 -Navarre,1.18 -Ceuta,84.67 -Castile an…,1.78 -Ceuta
max_3,83.6 -Castile an…,0.8 -Balearic I…,84.2 -Castile an…,0.77 -Murcia,84.66 -Navarre,1.39 -Aragon
max_4,83.45 -Navarre,0.8 -Cantabria,83.98 -Basque Cou…,0.66 -Basque Cou…,84.64 -Basque Cou…,1.38 -Madrid
max_5,83.35 -Basque Cou…,0.73 -Madrid,83.88 -Aragon,0.65 -Madrid,84.24 -Aragon,1.29 -Basque Cou…
Spain,– 82.91 –,– 0.62 –,– 83.53 –,– 0.48 –,– 84.01 –,– 1.1 –
min_5,82.11 -Asturias,0.5 -Andalusia,82.62 -Murcia,0.29 -Estremadura,83.13 -Estremadura,0.92 -Murcia
min_4,81.86 -Canary Isl…,0.49 -Estremadura,82.42 -Canary Isl…,0.28 -Castilla L…,82.82 -Andalusia,0.86 -Balearic I…
min_3,81.7 -Andalusia,0.29 -Castilla L…,82.2 -Andalusia,0.11 -Navarre,82.79 -Canary Isl…,0.78 -Estremadura
min_2,80.03 -Melilla,0.15 -Murcia,80.67 -Melilla,0.07 -La Rioja,82.66 -Melilla,0.57 -Castilla L…


In [15]:
mal.min_and_max_values(df[['m_2014', 'm_2019', 'm_2024', 'f_2014', 'f_2019', 'f_2024', 'f-m_2014', 'f-m_2019', 'f-m_2024']],
                       row_center=['Spain'], nmb=5, max_lng=11)

Number of records: 20


,m_2014,m_2019,m_2024,f_2014,f_2019,f_2024,f-m_2014,f-m_2019,f-m_2024
max,81.44 -Madrid,82.33 -Madrid,83.07 -Madrid,86.95 -La Rioja,87.19 -Madrid,87.74 -Madrid,6.7 -Ceuta,5.93 -Estremadura,5.79 -Asturias
max_2,80.82 -Castile an…,82.1 -Navarre,82.08 -Castile an…,86.59 -Madrid,86.94 -Castile an…,87.33 -Navarre,6.28 -Cantabria,5.78 -Asturias,5.63 -Galicia
max_3,80.69 -La Rioja,81.52 -Castile an…,81.96 -Navarre,86.44 -Castile an…,86.92 -Navarre,87.28 -Castile an…,6.26 -La Rioja,5.73 -Galicia,5.63 -Estremadura
max_4,80.68 -Castilla L…,81.26 -Aragon,81.91 -Basque Cou…,86.26 -Navarre,86.76 -Basque Cou…,87.2 -Basque Cou…,6.22 -Galicia,5.71 -Basque Cou…,5.61 -Cantabria
max_5,80.63 -Navarre,81.16 -Castilla L…,81.71 -Aragon,86.12 -Basque Cou…,86.62 -La Rioja,86.86 -Galicia,6.08 -Asturias,5.53 -La Rioja,5.37 -Navarre
Spain,– 80.1 –,– 80.78 –,– 81.38 –,– 85.63 –,– 86.19 –,– 86.53 –,– 5.53 –,– 5.41 –,– 5.15 –
min_5,79.28 -Canary Isl…,79.87 -Asturias,80.36 -Estremadura,84.88 -Murcia,85.17 -Murcia,85.81 -Balearic I…,5.18 -Canary Isl…,4.88 -Castilla L…,4.9 -Castilla L…
min_4,78.99 -Asturias,79.84 -Canary Isl…,80.35 -Asturias,84.46 -Canary Isl…,85.02 -Canary Isl…,85.42 -Andalusia,5.15 -Madrid,4.86 -Madrid,4.75 -Balearic I…
min_3,78.97 -Andalusia,79.5 -Andalusia,80.18 -Andalusia,84.4 -Andalusia,84.85 -Andalusia,85.2 -Melilla,5.11 -Comunidad …,4.82 -Navarre,4.67 -Madrid
min_2,77.67 -Melilla,78.36 -Ceuta,80.07 -Melilla,83.32 -Ceuta,83.14 -Melilla,84.99 -Canary Isl…,4.87 -Murcia,4.8 -Balearic I…,4.45 -Canary Isl…


<br />
<br />

In [17]:
dd_replacement = {
    'Spain'   : {'es': ('España', ''), 'en': ('Spain on average', ''), 'ru': ('Испания в среднем', '')},
    'Andalusia'   : {'es': ('Andalucía', 'Andalucía'), 'en': ('Andalusia', 'Andalusia'), 'ru': ('Андалу́си́я (Андалу́зия)', 'Андалусия')},
    'Aragon'   : {'es': ('Aragón', 'Aragón'), 'en': ('Aragon', 'Aragon'), 'ru': ('Араго́н', 'Арагон')},
    'Asturias'   : {'es': ('Asturias', 'Asturias'), 'en': ('Asturias', 'Asturias'), 'ru': ('Асту́рия', 'Астурия')},
    'Balearic Islands'   : {'es': ('Islas Baleares', 'Islas Baleares'), 'en': ('Balearic Islands', 'Balearic Islands'), 'ru': ('Балеа́рские острова', 'Балеарские острова')},
    'Basque Country'   : {'es': ('País Vasco', 'País Vasco'), 'en': ('Basque Country', 'Basque Country (autonomous community)'), 'ru': ('Страна Ба́сков', 'Страна Басков')},
    'Canary Islands'   : {'es': ('Canarias', 'Canarias'), 'en': ('Canary Islands', 'Canary Islands'), 'ru': ('Кана́рские острова', 'Канарские острова')},
    'Cantabria'   : {'es': ('Cantabria', 'Cantabria'), 'en': ('Cantabria', 'Cantabria'), 'ru': ('Канта́брия', 'Кантабрия')},
    'Castile and León'   : {'es': ('Castilla y León', 'Castilla y León'), 'en': ('Castile and León', 'Castile and León'), 'ru': ('Касти́лия-Лео́н', 'Кастилия-Леон')},    # or '(Касти́лия и Лео́н)'
    'Castilla La Mancha'   : {'es': ('Castilla-La Mancha', 'Castilla-La Mancha'), 'en': ('Castilla–La Mancha', 'Castilla–La Mancha'), 'ru': ('Касти́лия-Ла-Ма́нча', 'Кастилия-Ла-Манча')},
    'Catalonia'   : {'es': ('Cataluña', 'Cataluña'), 'en': ('Catalonia', 'Catalonia'), 'ru': ('Катало́ния', 'Каталония')},
    'Ceuta'   : {'es': ('Ceuta', 'Ceuta'), 'en': ('Ceuta (auton. city)', 'Ceuta'), 'ru': ('Сеу́та (автон. город)', 'Сеута')},
    'Comunidad Valenciana'   : {'es': ('Comunidad Valenciana', 'Comunidad Valenciana'), 'en': ('Valencia', 'Valencian Community'), 'ru': ('Валенси́йское сообщество', 'Валенсия (автономное сообщество)')},
    'Estremadura'   : {'es': ('Extremadura', 'Extremadura'), 'en': ('Extremadura', 'Extremadura'), 'ru': ('Эстремаду́ра', 'Эстремадура')},
    'Galicia'   : {'es': ('Galicia', 'Galicia'), 'en': ('Galicia', 'Galicia (Spain)'), 'ru': ('Гали́сия', 'Галисия')},
    'La Rioja'   : {'es': ('La Rioja', 'La Rioja (España)'), 'en': ('La Rioja', 'La Rioja'), 'ru': ('Рио́ха (Ла-Рио́ха)', 'Риоха')},
    'Madrid'   : {'es': ('Madrid', 'Comunidad de Madrid'), 'en': ('Madrid', 'Community of Madrid'), 'ru': ('Мадри́д', 'Мадрид (автономное сообщество)')},
    'Melilla'   : {'es': ('Melilla (ciudad autónoma)', 'Melilla'), 'en': ('Melilla (auton. city)', 'Melilla'), 'ru': ('Мели́лья (автон. город)', 'Мелилья')},
    'Murcia'   : {'es': ('Murcia', 'Región de Murcia'), 'en': ('Murcia', 'Region of Murcia'), 'ru': ('Му́рсия', 'Мурсия (автономное сообщество)')},
    'Navarre'   : {'es': ('Navarra', 'Navarra'), 'en': ('Navarre', 'Navarre'), 'ru': ('Нава́рра', 'Наварра (автономное сообщество)')}
}

In [18]:
# create code for placing info in Wikipedia
def create_table_v1(df, file_header, lang='ru', is_extended=False):

    def if_value(x, prec=2):
        return '—' if math.isnan(x) else \
               f"{x:0.{prec}f}"  if x>=0 else \
               f"−{-x:0.{prec}f}"                #"{x:0.{prec}f}".format(x, prec)
    
    def chval(x, prec=2, *, add_par=''):  # change_value
        return f'style="background:#fffae0;{add_par}"| —' if math.isnan(x) else \
               f'style="background:#fffae0;color:darkgreen;{add_par}"| {x:0.{prec}f}' if x>0 else \
               f'style="background:#fffae0;color:crimson;{add_par}"| −{-x:0.{prec}f}' if x<0 else \
               f'style="background:#fffae0;color:darkgray;{add_par}"| {x:0.{prec}f}'
    
    def chval_bold(x, prec=2, *, add_par=''):  # change_value
        return ' —' if math.isnan(x) else \
               f'style="background:#fffae0;color:darkgreen;{add_par}"| \'\'\'{x:0.{prec}f}\'\'\'' if x>0 else \
               f'style="background:#fffae0;color:crimson;{add_par}"| \'\'\'−{-x:0.{prec}f}\'\'\'' if x<0 else \
               f'style="background:#fffae0;color:darkgray;{add_par}"| \'\'\'{x:0.{prec}f}\'\'\''
    
    with open('design/' + file_header, mode='r', encoding="utf-8") as fh:
        table_header = fh.read()

    st = ''
    for i in range(len(df)):
        ser = df.iloc[i]
        if ser.name == 'Spain':
             st += '\n' + '|-class=static-row-header\n' + \
                  f'| \'\'\'{dd_replacement[ser.name][lang][0]}\'\'\' ' + \
                  f'||style="background:#e0ffd8;"| \'\'\'{if_value(ser["2024"])}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| \'\'\'{if_value(ser["m_2024"])}\'\'\' ' + \
                  f'||style="background:#fee7f6;"| \'\'\'{if_value(ser["f_2024"])}\'\'\' ' + \
                  f'||style="background:#fff8dc;"| \'\'\'{if_value(ser["f-m_2024"])}\'\'\' ' + \
                  f'||style="border-left-width:2px;"| \'\'\'{if_value(ser["2014"])}\'\'\' ' + \
                  f'||{chval_bold(ser["2014→2019"])} ' + \
                  f'|| \'\'\'{if_value(ser["2019"])}\'\'\' ' + \
                  f'||{chval_bold(ser["2019→2020"])} ' + \
                  f'|| \'\'\'{if_value(ser["2020"])}\'\'\' ' + \
                  f'||{chval_bold(ser["2020→2021"])} ' + \
                  f'|| \'\'\'{if_value(ser["2021"])}\'\'\' ' + \
                  f'||{chval_bold(ser["2021→2022"])} ' + \
                  f'|| \'\'\'{if_value(ser["2022"])}\'\'\' ' + \
                  f'||{chval_bold(ser["2022→2023"])} ' + \
                  f'|| \'\'\'{if_value(ser["2023"])}\'\'\' ' + \
                  f'||{chval_bold(ser["2023→2024"])} ' + \
                  f'||style="background:#e0ffd8;"| \'\'\'{if_value(ser["2024"])}\'\'\' ' + \
                  f'||{chval_bold(ser["2014→2024"], add_par="border-left-width:2px;")}'
        else:
            name_link = dd_replacement[ser.name][lang][1]
            name_visible = dd_replacement[ser.name][lang][0]
            name_inserted = name_link if name_link == name_visible else f"{name_link}|{name_visible}"
            st += '\n' + '|-\n' + \
                  f'| [[{name_inserted}]] ' + \
                  f'||style="background:#e0ffd8;"| \'\'\'{if_value(ser["2024"])}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| {if_value(ser["m_2024"])} ' + \
                  f'||style="background:#fee7f6;"| {if_value(ser["f_2024"])} ' + \
                  f'||style="background:#fff8dc;"| {if_value(ser["f-m_2024"])} ' + \
                  f'||style="border-left-width:2px;"| {if_value(ser["2014"])} ' + \
                  f'||{chval(ser["2014→2019"])} ' + \
                  f'|| {if_value(ser["2019"])} ' + \
                  f'||{chval(ser["2019→2020"])} ' + \
                  f'|| {if_value(ser["2020"])} ' + \
                  f'||{chval(ser["2020→2021"])} ' + \
                  f'|| {if_value(ser["2021"])} ' + \
                  f'||{chval(ser["2021→2022"])} ' + \
                  f'|| {if_value(ser["2022"])} ' + \
                  f'||{chval(ser["2022→2023"])} ' + \
                  f'|| {if_value(ser["2023"])} ' + \
                  f'||{chval(ser["2023→2024"])} ' + \
                  f'||style="background:#e0ffd8;"| \'\'\'{if_value(ser["2024"])}\'\'\' ' + \
                  f'||{chval(ser["2019→2024"], add_par="border-left-width:2px;")}'

    if lang == 'ru':
        st = re.sub('(?<=\\d)\\.(?=\\d)', ',', st)  # replace . to comma, if this . is between two digits

    st = table_header + st + '\n|}'
    
    # gray color for missing values
    st = st.replace(';"|—', ';color:silver;"|—')

    return st


table_code = create_table_v1(df, file_header='Spanish_header_ru -2024 -v1.txt', lang='ru')

# write the code to file
with open('output/Table code for Spanish autonomous communities -ru -v1.txt', 'w', encoding="utf-8") as fh:
    fh.write(table_code)

In [19]:
table_code = create_table_v1(df, file_header='Spanish_header_en -2024 -v1.txt', lang='en')

# write the code to file
with open('output/Table code for Spanish autonomous communities -en -v1.txt', 'w', encoding="utf-8") as fh:
    fh.write(table_code)

<br />
<br />

In [21]:
# create code for placing info in Wikipedia
def create_table_v2(df, file_header, lang='ru', is_extended=False):

    def if_value(x, prec=2):
        return '—' if math.isnan(x) else \
               f"{x:0.{prec}f}"  if x>=0 else \
               f"−{-x:0.{prec}f}"                #"{x:0.{prec}f}".format(x, prec)
    
    def chval(x, prec=2, *, add_par=''):  # change_value
        return f'style="background:#fffae0;{add_par}"| —' if math.isnan(x) else \
               f'style="background:#fffae0;color:darkgreen;{add_par}"| {x:0.{prec}f}' if x>0 else \
               f'style="background:#fffae0;color:crimson;{add_par}"| −{-x:0.{prec}f}' if x<0 else \
               f'style="background:#fffae0;color:darkgray;{add_par}"| {x:0.{prec}f}'
    
    def chval_bold(x, prec=2, *, add_par=''):  # change_value
        return ' —' if math.isnan(x) else \
               f'style="background:#fffae0;color:darkgreen;{add_par}"| \'\'\'{x:0.{prec}f}\'\'\'' if x>0 else \
               f'style="background:#fffae0;color:crimson;{add_par}"| \'\'\'−{-x:0.{prec}f}\'\'\'' if x<0 else \
               f'style="background:#fffae0;color:darkgray;{add_par}"| \'\'\'{x:0.{prec}f}\'\'\''
    
    with open('design/' + file_header, mode='r', encoding="utf-8") as fh:
        table_header = fh.read()

    st = ''
    for i in range(len(df)):
        ser = df.iloc[i]
        if ser.name == 'Spain':
             st += '\n' + '|-class=static-row-header\n' + \
                  f'| \'\'\'{dd_replacement[ser.name][lang][0]}\'\'\' ' + \
                  f'||style="background:#e0ffd8;"| \'\'\'{if_value(ser["2014"])} ' + \
                  f'||style="background:#eaf3ff;"| \'\'\'{if_value(ser["m_2014"])} ' + \
                  f'||style="background:#fee7f6;"| \'\'\'{if_value(ser["f_2014"])} ' + \
                  f'||style="background:#fffae0;"| \'\'\'{if_value(ser["f-m_2014"])}\'\'\' ' + \
                  f'||{chval_bold(ser["2014→2019"], add_par="border-left-width:2px;padding-right:1.5ex;")} ' + \
                  f'||style="border-left-width:2px;background:#e0ffd8;"| \'\'\'{if_value(ser["2019"])}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| \'\'\'{if_value(ser["m_2019"])}\'\'\' ' + \
                  f'||style="background:#fee7f6;"| \'\'\'{if_value(ser["f_2019"])}\'\'\' ' + \
                  f'||style="background:#fffae0;"| \'\'\'{if_value(ser["f-m_2019"])}\'\'\' ' + \
                  f'||{chval_bold(ser["2019→2024"], add_par="border-left-width:2px;padding-right:1.5ex;")} ' + \
                  f'||style="border-left-width:2px;background:#e0ffd8;"| \'\'\'{if_value(ser["2024"])}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| \'\'\'{if_value(ser["m_2024"])}\'\'\' ' + \
                  f'||style="background:#fee7f6;"| \'\'\'{if_value(ser["f_2024"])}\'\'\' ' + \
                  f'||style="background:#fffae0;"| \'\'\'{if_value(ser["f-m_2024"])}\'\'\' ' + \
                  f'||{chval_bold(ser["2014→2024"], add_par="border-left-width:2px;padding-right:1.5ex;")}'
        else:
            name_link = dd_replacement[ser.name][lang][1]
            name_visible = dd_replacement[ser.name][lang][0]
            name_inserted = name_link if name_link == name_visible else f"{name_link}|{name_visible}"
            st += '\n' + '|-\n' + \
                  f'| [[{name_inserted}]] ' + \
                  f'||style="background:#e0ffd8;"| {if_value(ser["2014"])} ' + \
                  f'||style="background:#eaf3ff;"| {if_value(ser["m_2014"])} ' + \
                  f'||style="background:#fee7f6;"| {if_value(ser["f_2014"])} ' + \
                  f'||style="background:#fffae0;"| {if_value(ser["f-m_2014"])} ' + \
                  f'||{chval(ser["2014→2019"], add_par="border-left-width:2px;padding-right:1.5ex;")} ' + \
                  f'||style="border-left-width:2px;background:#e0ffd8;"| {if_value(ser["2019"])} ' + \
                  f'||style="background:#eaf3ff;"| {if_value(ser["m_2019"])} ' + \
                  f'||style="background:#fee7f6;"| {if_value(ser["f_2019"])} ' + \
                  f'||style="background:#fffae0;"| {if_value(ser["f-m_2019"])} ' + \
                  f'||{chval(ser["2019→2024"], add_par="border-left-width:2px;padding-right:1.5ex;")} ' + \
                  f'||style="border-left-width:2px;background:#e0ffd8;"| {if_value(ser["2024"])} ' + \
                  f'||style="background:#eaf3ff;"| {if_value(ser["m_2024"])} ' + \
                  f'||style="background:#fee7f6;"| {if_value(ser["f_2024"])} ' + \
                  f'||style="background:#fffae0;"| {if_value(ser["f-m_2024"])} ' + \
                  f'||{chval(ser["2014→2024"], add_par="border-left-width:2px;padding-right:1.5ex;")}'

    if lang == 'ru':
        st = re.sub('(?<=\\d)\\.(?=\\d)', ',', st)  # replace . to comma, if this . is between two digits
        st = st.replace('padding-right:1,5ex;', 'padding-right:1.5ex;')

    st = table_header + st + '\n|}'
    
    # gray color for missing values
    st = st.replace(';"|—', ';color:silver;"|—')

    return st


table_code = create_table_v2(df, file_header='Spanish_header_ru -2024 -v2.txt', lang='ru')

# write the code to file
with open('output/Table code for Spanish autonomous communities -ru -v2.txt', 'w', encoding="utf-8") as fh:
    fh.write(table_code)

In [22]:
table_code = create_table_v2(df, file_header='Spanish_header_en -2024 -v2.txt', lang='en')

# write the code to file
with open('output/Table code for Spanish autonomous communities -en -v2.txt', 'w', encoding="utf-8") as fh:
    fh.write(table_code)